In [44]:
import polars as pl
from data_engineering.flights.flight_type import get_flights
from datetime import date
df = pl.read_parquet('/Volumes/T2-SSD/planequery/data/tests/test_pia/pia_tester/test_pia_pia_tester_20260821_004941/pia_mapping_audit.parquet', glob=False)

In [45]:
df = df.unique(subset="observed_icao").filter(pl.col("predicted_icao").is_not_null() & pl.col("correct").is_not_null())
df = df.select("observed_icao", "predicted_icao", "real_icao", "correct")
df

observed_icao,predicted_icao,real_icao,correct
str,str,str,bool
"""a03edd""","""a92f04""","""a31ae0""",false
"""a03c1f""","""a3c53f""","""a21084""",false
"""a03f2c""","""a5002c""","""a5002c""",true
"""a03b1d""","""a6548b""","""a00529""",false
"""a0ff27""","""ac2749""","""a18db5""",false
…,…,…,…
"""a500c2""","""aae677""","""a520e2""",false
"""a0fee1""","""a7c365""","""a7c365""",true
"""a0398c""","""a7dabe""","""a18884""",false


In [46]:
df_flights = get_flights(date(2024,1,1), date(2026,7,1))

In [47]:
observed_icaos = df["observed_icao"].to_list()
predicted_real_icaos = df["predicted_icao"].to_list()
real_icaos = df["real_icao"].to_list()
correct = df["correct"].to_list()
n_switches_list = []
for observed_icao, predicted_real_icao, real_icao, is_correct in zip(observed_icaos, predicted_real_icaos, real_icaos, correct):
    df_flights_filtered = df_flights.filter(pl.col("icao").is_in([predicted_real_icao, observed_icao])).sort("takeoff_time")
    icao_sequence = df_flights_filtered["icao"].to_list()
    n_switches = sum(1 for i in range(1, len(icao_sequence)) if icao_sequence[i] != icao_sequence[i - 1])
    n_switches_list.append(n_switches)

df_mapping = df.with_columns(pl.Series(n_switches_list).alias("n_switches"))
df_mapping

observed_icao,predicted_icao,real_icao,correct,n_switches
str,str,str,bool,i64
"""a03edd""","""a92f04""","""a31ae0""",false,1
"""a03c1f""","""a3c53f""","""a21084""",false,1
"""a03f2c""","""a5002c""","""a5002c""",true,1
"""a03b1d""","""a6548b""","""a00529""",false,1
"""a0ff27""","""ac2749""","""a18db5""",false,1
…,…,…,…,…
"""a500c2""","""aae677""","""a520e2""",false,1
"""a0fee1""","""a7c365""","""a7c365""",true,1
"""a0398c""","""a7dabe""","""a18884""",false,1


In [48]:
df_mapping.sort("n_switches", descending=True)

observed_icao,predicted_icao,real_icao,correct,n_switches
str,str,str,bool,i64
"""a03bff""","""a11784""","""a11784""",true,9
"""a0feb4""","""a8f2d6""","""a8f2d6""",true,2
"""a03edd""","""a92f04""","""a31ae0""",false,1
"""a03c1f""","""a3c53f""","""a21084""",false,1
"""a03f2c""","""a5002c""","""a5002c""",true,1
…,…,…,…,…
"""a500c2""","""aae677""","""a520e2""",false,1
"""a0fee1""","""a7c365""","""a7c365""",true,1
"""a0398c""","""a7dabe""","""a18884""",false,1


In [43]:
COLUMNS = ["icao", "pia", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "callsign"]
with pl.Config(tbl_rows=-1):
    display(df_flights.filter(pl.col("icao").is_in(["a03c06", "a97b71"])).select(COLUMNS).sort("takeoff_time"))

icao,pia,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,callsign
str,bool,datetime[ms],str,datetime[ms],str,str
"""a03c06""",true,2026-01-05 14:55:12.830,"""KSAV""",2026-01-05 15:54:11.060,"""KPBI""","""XAA9036"""
"""a97b71""",false,2026-01-21 16:20:01.180,"""KPBI""",2026-01-21 18:27:50,"""KISP""","""N71TV"""
"""a97b71""",false,2026-01-26 22:44:42.310,"""KISP""",2026-01-26 23:59:58.350,"""KEWN""","""N71TV"""
"""a97b71""",false,2026-01-27 00:00:18.070,"""KECG""",2026-01-27 01:38:38.100,"""KPBI""","""N71TV"""
"""a03c06""",true,2026-01-27 13:04:02.200,"""KPBI""",2026-01-27 15:36:47.110,"""KAUS""","""XAA6810"""
"""a03c06""",true,2026-01-27 16:40:58.430,"""KAUS""",2026-01-27 17:50:50.430,"""KPNS""","""XAA6490"""
"""a03c06""",true,2026-01-27 18:24:27.490,"""KPNS""",2026-01-27 19:36:37.520,"""KPBI""","""XAA282"""
"""a03c06""",true,2026-02-01 21:55:32.760,"""KPBI""",2026-02-01 23:16:18.250,"""KCLT""","""XAA1304"""
"""a03c06""",true,2026-02-01 23:50:20.810,"""KCLT""",2026-02-01 23:59:59.470,"""NC02""","""XAA613"""
